# Chunking Strategies

## 2. Install the required packages

In [1]:
!uv pip install -q langchain-text-splitters pypdf openai tiktoken sentence-transformers matplotlib pandas python-dotenv

In [2]:
import os, re, json, statistics
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)

## 3. Load environment variables for API keys

In [3]:
load_dotenv()   # reads the .env file

if os.getenv("OPENAI_API_KEY"):
    print("API key found")
else:
    print("No API key found. Check your .env file (it is needed for transcription).")

API key found


## 4. Load the podcast audio file

In [4]:
AUDIO_PATH = Path("The_Blueprint_For_Trustworthy_AI.m4a")

print("Current working folder:", Path.cwd())
if AUDIO_PATH.exists():
    print(f"Audio found: {AUDIO_PATH.name} ({AUDIO_PATH.stat().st_size / 1_000_000:.1f} MB)")
else:
    print("Audio NOT FOUND. Move the file into the folder shown above.")

Current working folder: /Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/course work/week-3 /lab-chunking-strategies
Audio found: The_Blueprint_For_Trustworthy_AI.m4a (30.2 MB)


## 5. Transcribe the audio file (save it to a local file)

In [5]:
import subprocess, glob

TRANSCRIPT_PATH = Path("transcript.txt")

def transcribe_audio(audio_path, out_path, model="whisper-1", piece_seconds=600):
    """Transcribe audio with the OpenAI API and save the text. Skips all work if out_path already exists."""
    out = Path(out_path)
    if out.exists() and out.stat().st_size > 0:
        print(f"{out} already exists -> skipping transcription")
        return out.read_text(encoding="utf-8")

    # 1) shrink and cut the audio: mono, 64 kbps mp3, 10-minute pieces
    Path("_pieces").mkdir(exist_ok=True)
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(audio_path), "-ac", "1", "-b:a", "64k",
                    "-f", "segment", "-segment_time", str(piece_seconds), "-reset_timestamps", "1",
                    "_pieces/piece_%02d.mp3"], check=True)

    # 2) transcribe the pieces in order
    from openai import OpenAI
    client = OpenAI()                                  # uses OPENAI_API_KEY from the environment
    texts = []
    for f in sorted(glob.glob("_pieces/piece_*.mp3")):
        with open(f, "rb") as fh:
            texts.append(client.audio.transcriptions.create(model=model, file=fh).text)
        print(f, "done")

    text = " ".join(texts)
    out.write_text(text, encoding="utf-8")             # save to a local file
    return text

podcast_text = transcribe_audio(AUDIO_PATH, TRANSCRIPT_PATH)
print(f"Podcast transcript length: {len(podcast_text)} chars")
print(podcast_text[:200])

transcript.txt already exists -> skipping transcription
Podcast transcript length: 16461 chars
So imagine for a second you're driving across, I don't know, a massive suspension bridge. OK. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview t


## 6. Load the PDF document (from a URL or a local file)

In [6]:
PDF_PATH = Path("ai_hleg_ethics_guidelines_for_trustworthy_ai-en.pdf")

def load_pdf(path=PDF_PATH):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

pdf_text = load_pdf()
print(f"PDF length: {len(pdf_text)} chars")
print(pdf_text[:200])

PDF length: 156411 chars
 
 
 
 
 
 
INDEPENDENT 
HIGH-LEVEL EXPERT GROUP ON 
ARTIFICIAL INTELLIGENCE 
SET UP BY THE EUROPEAN COMMISSION 
 
 
 
 
 
 
ETHICS GUIDELINES 
FOR TRUSTWORTHY AI 
 
 
 
 
 
 
ETHICS GUIDELINES FOR TR


# Step 2 — Implement Fixed-Size Chunking

## 1. fixed-size chunks for both documents with `CharacterTextSplitter`.

In [7]:
splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50)

splitter.split_text(podcast_text)

["So imagine for a second you're driving across, I don't know, a massive suspension bridge. OK. You don't pull over halfway across, get out and demand to see the blueprints, right? You don't interview the welding crew. No. You just, you trust it. You just drive. You trust the bridge. You trust the engineering standards, the inspections, the laws that say this thing won't fail. Right. It's trust in the infrastructure. It's invisible, but it's there. Exactly. But now let's switch gears. Think about",
 "Exactly. But now let's switch gears. Think about the algorithm that just denied your mortgage application or the A.I. system scanning your face at the airport. Do you have that same trust? And I mean, honestly, should you? That is the defining question of our decade, I think. We've moved past the wow phase of A.I. and straight into the wait a minute phase. Today we are digging into the blueprint for that trust. We're unpacking a document that is arguably the Magna Carta for ethical computi

## Analysis questions

**1. Does fixed-size chunking break sentences in the middle?**

In [8]:
import statistics

def chunk_stats(chunks, label=""):
    sizes = [len(c) for c in chunks]
    clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
    pct_clean = 100 * clean_end / len(chunks)
    print(f"{label}: n={len(chunks)}, avg={statistics.mean(sizes):.1f}, "
          f"min={min(sizes)}, max={max(sizes)}, clean_end={clean_end} ({pct_clean:.1f}%)")
    return sizes, clean_end, pct_clean

# Look at a boundary directly
splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50)
podcast_chunks = splitter.split_text(podcast_text)

for i in (0, 1, 2):
    print(f"chunk {i} end:    ...{podcast_chunks[i][-50:]!r}")
    print(f"chunk {i+1} start: {podcast_chunks[i+1][:50]!r}...\n")

# Count how many chunks end "cleanly" (on sentence punctuation), varying chunk_size / overlap
print(f"{'size':>6} {'overlap':>8} {'n_chunks':>9} {'clean_end':>10} {'pct_clean':>10}")
for chunk_size in (500, 1000, 2000):
    for overlap in (0, 50, 100):
        splitter = CharacterTextSplitter(separator="", chunk_size=chunk_size, chunk_overlap=overlap)
        chunks = splitter.split_text(podcast_text)
        clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
        pct_clean = 100 * clean_end / len(chunks)
        print(f"{chunk_size:6d} {overlap:8d} {len(chunks):9d} {clean_end:10d} {pct_clean:9.1f}%")


chunk 0 end:    ...". Exactly. But now let's switch gears. Think about"
chunk 1 start: "Exactly. But now let's switch gears. Think about t"...

chunk 1 end:    ...' arguably the Magna Carta for ethical computing, t'
chunk 2 start: 'arguably the Magna Carta for ethical computing, th'...

chunk 2 end:    ..."cally the Stone Age. We didn't have the generative"
chunk 3 start: "cally the Stone Age. We didn't have the generative"...

  size  overlap  n_chunks  clean_end  pct_clean
   500        0        33          3       9.1%
   500       50        37          2       5.4%
   500      100        41          3       7.3%
  1000        0        17          1       5.9%
  1000       50        18          2      11.1%
  1000      100        19          3      15.8%
  2000        0         9          1      11.1%
  2000       50         9          1      11.1%
  2000      100         9          2      22.2%


✍️ **Answer — Q1 (sentence breaks):**

Yes, fixed-size chunking breaks sentences in the middle almost every time. Looking at the printed
boundaries above, chunk 0 ends mid-sentence ("...switch gears. Think about") and chunk 1 starts
exactly where it was cut off ("Exactly. But now let's switch gears. Think about t..."), with no
regard for the sentence or word boundary.

Counting it: with `chunk_size=500, chunk_overlap=50` only **2 of 37** podcast chunks (5.4%) end
cleanly on `.`, `?` or `!`. The percentage changes with chunk size and overlap, but stays low
across the board — from **9.1%** (size 500, overlap 0) down to **5.4%** (size 500, overlap 50),
up to **22.2%** at the largest size/overlap combo tried (size 2000, overlap 100). Bigger chunks
and more overlap give slightly more chances to land on a sentence end by coincidence, but the
splitter never actually looks for sentence boundaries — it just counts characters.


**2. How does it handle paragraph boundaries?**

In [9]:
results = {}
for sep_name, sep in (("empty", ""), ("space", " "), ("default", "\n\n")):
    for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
        splitter = CharacterTextSplitter(separator=sep, chunk_size=500, chunk_overlap=50)
        chunks = splitter.split_text(text)
        sizes = [len(c) for c in chunks]
        over_limit = sum(1 for s in sizes if s > 500)
        results[(sep_name, label)] = (len(chunks), statistics.mean(sizes), min(sizes), max(sizes), over_limit)
        print(f"sep={sep_name:8s} text={label:8s} n={len(chunks):4d} "
              f"avg={statistics.mean(sizes):7.1f} min={min(sizes):5d} max={max(sizes):5d} "
              f"over_chunk_size={over_limit}")

print()
print("podcast '\\n\\n' count:", podcast_text.count("\n\n"))
print("pdf '\\n\\n' count:    ", pdf_text.count("\n\n"))


sep=empty    text=podcast  n=  37 avg=  493.1 min=  261 max=  500 over_chunk_size=0


sep=empty    text=pdf      n= 348 avg=  498.0 min=  258 max=  500 over_chunk_size=0
sep=space    text=podcast  n=  37 avg=  490.8 min=  251 max=  500 over_chunk_size=0
sep=space    text=pdf      n= 342 avg=  495.4 min=  273 max=  500 over_chunk_size=0
sep=default  text=podcast  n=   1 avg=16461.0 min=16461 max=16461 over_chunk_size=1


Created a chunk of size 28061, which is longer than the specified 500


Created a chunk of size 29967, which is longer than the specified 500


Created a chunk of size 23694, which is longer than the specified 500


sep=default  text=pdf      n=   4 avg=39093.0 min=23693 max=74680 over_chunk_size=4

podcast '\n\n' count: 0
pdf '\n\n' count:     3


In [10]:
bonus = {}
# A single separator can't be ".", "!" and "?" at once -- CharacterTextSplitter also
# accepts a REGEX as the separator (is_separator_regex=True). "[.!?\n]\s+" matches a
# period, question mark, exclamation mark, OR a line break, followed by whitespace --
# the "\n" is added so that headings/list items (which have no sentence punctuation
# at all) also count as a valid split point.
# keep_separator="end" keeps the matched punctuation+space attached to the end of
# each sentence (otherwise CharacterTextSplitter has a known quirk: with a regex
# separator it can literally re-insert the pattern text, or with a lookaround
# pattern it drops the separator and glues sentences back together with no space).
for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
    splitter = CharacterTextSplitter(
        separator=r"[.!?\n]\s+",
        is_separator_regex=True,
        keep_separator="end",
        chunk_size=500,
        chunk_overlap=50,
    )
    chunks = splitter.split_text(text)
    sizes = [len(c) for c in chunks]
    clean_end = sum(1 for c in chunks if c.endswith("\n") or c.rstrip().endswith((".", "?", "!")))
    over_limit = sum(1 for s in sizes if s > 500)
    bonus[label] = chunks
    print(f"text={label:8s} n={len(chunks):4d} avg={statistics.mean(sizes):7.1f} "
          f"min={min(sizes):5d} max={max(sizes):5d} clean_end={clean_end:4d} "
          f"({100*clean_end/len(chunks):.1f}%) over_chunk_size={over_limit}")

print()
for c in bonus["podcast"][:3]:
    print(repr(c[-60:]))


Created a chunk of size 602, which is longer than the specified 500


Created a chunk of size 546, which is longer than the specified 500


Created a chunk of size 518, which is longer than the specified 500


Created a chunk of size 1021, which is longer than the specified 500


Created a chunk of size 501, which is longer than the specified 500


...

Created a chunk of size 856, which is longer than the specified 500


Created a chunk of size 735, which is longer than the specified 500


Created a chunk of size 512, which is longer than the specified 500


Created a chunk of size 683, which is longer than the specified 500


text=podcast  n=  37 avg=  463.9 min=  339 max=  494 clean_end=  37 (100.0%) over_chunk_size=0
text=pdf      n= 385 avg=  404.9 min=  136 max=  959 clean_end= 357 (92.7%) over_chunk_size=9

"isible, but it's there. Exactly. But now let's switch gears."
'ase. Today we are digging into the blueprint for that trust.'
'yone listening, this document, it went public in April 2019.'


✍️ **Answer — Q2 (paragraph boundaries, plus comparing to a word separator):**

With the **default separator** (`"\n\n"`), `CharacterTextSplitter` only cuts where a double
newline exists — it never splits inside a "paragraph," even if that paragraph is far bigger than
`chunk_size`. The podcast transcript has **0** occurrences of `\n\n` (it's one continuous block
of speech-to-text), so it comes back as **1 single chunk of 16,461 characters** — 33x bigger than
the requested `chunk_size=500`. The PDF has only **3** occurrences of `\n\n`, so it becomes
**4 chunks**, with a max chunk of **74,680 characters** — again far above the 500-character
target. Both texts end up with **100% of chunks over `chunk_size`**, because the separator is
only a candidate split point — `chunk_size` is never enforced if that character never shows up.

`separator=""` and `separator=" "` behave almost identically here (37/342-348 chunks, all ≤500
chars): they ignore structure and just cut by raw count (or by count that happens to land on a
space), so they reliably respect the size limit but can slice through words or sentences freely.

A real sentence ends with `.`, `!` **or** `?` — one literal separator can't match all three.
`CharacterTextSplitter` also accepts a **regex** as the separator (`is_separator_regex=True`), so
`separator=r"[.!?]\s+"` matches *any* of them followed by whitespace. With `keep_separator="end"`
(to avoid two quirks: a literal regex pattern getting re-inserted as text, or a lookaround pattern
eating the space between sentences), this gave **100% clean sentence endings** on the podcast
(37/37) but only partially fixed the PDF — because a PDF also has **headings and list items that
end with a line break and no punctuation at all**, which `[.!?]\s+` can't see as a boundary.

Adding `\n` to the class — `separator=r"[.!?\n]\s+"` (see the cell below) — treats a plain line
break as a valid split point too. This changes the PDF picture noticeably: it goes from
**14 chunks over 500 chars (max 2,744)** with `[.!?]\s+` down to **9 chunks over 500 chars (max
959)** with `[.!?\n]\s+` — because headings/short lines that previously got glued onto a long
run of body text until the next `.`/`!`/`?` now get cut off right away. The "clean ending" rate
on the PDF correspondingly rises from a `.`/`!`/`?`-only count to **92.7%** once a line-break
ending also counts as "clean."

**Takeaway:** `chunk_size` with `CharacterTextSplitter` is a *soft cap* that only holds when the
separator is frequent enough in the text, **and** actually present at every kind of boundary the
text has. `""`/`" "` are frequent but ignore meaning; `"\n\n"` alone is far too rare; `[.!?]\s+`
respects sentence boundaries but misses structural ones like headings; adding `\n` closes most of
that gap for a structured document like the PDF. None of this is free of tradeoffs: the regex
approaches still make false splits on abbreviations ("A.I. powerful" → "...A.I." / "powerful..."),
and `chunk_size` is still only approximate — exactly the tradeoff `RecursiveCharacterTextSplitter`
is built to manage, by falling back through a list of separators instead of committing to one.


**3. Which content type handles fixed-size chunking better?**

In [11]:
for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
    splitter = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_text(text)
    chunk_stats(chunks, label)


podcast: n=37, avg=493.1, min=261, max=500, clean_end=2 (5.4%)
pdf: n=348, avg=498.0, min=258, max=500, clean_end=11 (3.2%)


✍️ **Answer — Q3 (podcast vs. PDF):**

Side by side (`chunk_size=500, chunk_overlap=50, separator=""`): the podcast transcript yields
**37 chunks** (avg 493.1, min 261, max 500) with **5.4%** ending cleanly on sentence punctuation,
while the PDF yields **348 chunks** (avg 498.0, min 258, max 500) with only **3.2%** ending
cleanly.

Neither text handles fixed-size chunking *well*, but the PDF is somewhat worse for clean
sentence boundaries. That matches what the raw text looks like: the podcast is a transcribed
conversation with normal sentence punctuation throughout, so a few splits land on `.`/`?`/`!`
just by chance. The PDF, in contrast, is extracted body text full of headings, bullet lists,
footnotes and page-break artifacts (and almost no `\n\n`, as shown in Q2) — so sentence-ending
punctuation is sparser relative to its length, and `separator=""` slices through headings,
list items and footnote numbers just as readily as through prose.

**Bonus — does a smarter, boundary-aware splitter change the picture?** Yes, and the PDF is
where it matters most. With `[.!?]\s+` (sentence punctuation only), the podcast hits 100% clean
endings and stays fully under the size cap, while the PDF reaches 100% clean endings too but
still has **14 of 381 chunks over 500 chars (max 2,744)** — because headings/list items have no
sentence punctuation to split on, so the splitter runs straight through them into the next
paragraph. Adding `\n` to the separator — `[.!?\n]\s+` — targets exactly that gap: it drops the
PDF's over-size chunks from 14 to **9** (max down to 959) and lifts its clean-ending rate to
**92.7%**, specifically because it can now also stop at a heading or a short line, not just at a
full sentence. The podcast's numbers don't change (still 37 chunks, 100% clean, 0 over the cap),
because it has no headings or list structure to begin with — it's just continuous speech.

So the real difference between the two content types isn't really about "sentences" — it's about
**document structure**. The podcast is prose from end to end, so a sentence-only separator already
chunks it predictably. The PDF mixes prose with structural elements (headings, lists, tables)
that don't follow sentence grammar at all, so it needs separators that understand *both* kinds of
boundary to chunk reliably — exactly the reasoning behind `RecursiveCharacterTextSplitter`'s
ordered list of separators (`["\n\n", "\n", " ", ""]`) instead of one fixed rule.


# Step 3 — Implement Recursive Character Chunking

## 1. Recursive chunks for both documents with `RecursiveCharacterTextSplitter`.

`RecursiveCharacterTextSplitter` tries a **priority list** of separators instead of one fixed
rule: `["\n\n", "\n", " ", ""]` by default. It splits on the first separator that actually occurs
in the text, then recursively re-splits any piece still bigger than `chunk_size` using the *next*
separator in the list — only falling all the way to raw characters (`""`) if nothing else works.

In [12]:
print("default separator priority:", RecursiveCharacterTextSplitter()._separators)
print()

print(f"{'size':>6} {'text':>8} {'n_chunks':>9} {'clean_end':>10} {'over_size':>10}")
for chunk_size in (500, 1000, 2000):
    for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
        splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=50)
        chunks = splitter.split_text(text)
        sizes = [len(c) for c in chunks]
        clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
        over = sum(1 for s in sizes if s > chunk_size)
        print(f"{chunk_size:6d} {label:>8s} {len(chunks):9d} {clean_end:10d} {over:10d}")

# Boundary check for the podcast, same as Steps 2 and 4
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
recursive_podcast_chunks = splitter.split_text(podcast_text)
for c in recursive_podcast_chunks[:5]:
    print(repr(c[-30:]))


default separator priority: ['\n\n', '\n', ' ', '']

  size     text  n_chunks  clean_end  over_size
   500  podcast        37          4          0
   500      pdf       353         79          0
  1000  podcast        18          2          0
  1000      pdf       168         38          0
  2000  podcast         9          1          0
  2000      pdf        83         22          0
"et's switch gears. Think about"
'a Carta for ethical computing,'
" We didn't have the generative"
"tionalize that. We aren't just"
' Right. And they have to exist'


## Analysis questions

**1. Does recursive chunking preserve sentence boundaries better?**

In [13]:
# Default separators have no sentence-ending punctuation at all -- only "\n\n", "\n", " ", "".
# What if we add one? Two attempts, to show a real gotcha with `keep_separator`.

# Attempt A: add ". " etc. to the list, WITHOUT keep_separator (the default, False)
splitter_a = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
)
chunks_a = splitter_a.split_text(podcast_text)
clean_a = sum(1 for c in chunks_a if c.rstrip().endswith((".", "?", "!")))
print(f"A) custom separators, keep_separator=False (default): clean_end={clean_a}/{len(chunks_a)} "
      f"({100*clean_a/len(chunks_a):.1f}%)")
for c in chunks_a[:3]:
    print(" ", repr(c[-30:]))

# Attempt B: same separators, WITH keep_separator="end"
splitter_b = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""],
    keep_separator="end",
)
chunks_b = splitter_b.split_text(podcast_text)
clean_b = sum(1 for c in chunks_b if c.rstrip().endswith((".", "?", "!")))
print(f"\nB) same separators, keep_separator='end': clean_end={clean_b}/{len(chunks_b)} "
      f"({100*clean_b/len(chunks_b):.1f}%)")
for c in chunks_b[:3]:
    print(" ", repr(c[-30:]))


A) custom separators, keep_separator=False (default): clean_end=1/37 (2.7%)
  "ly. But now let's switch gears"
  'o the blueprint for that trust'
  ', it went public in April 2019'

B) same separators, keep_separator='end': clean_end=37/37 (100.0%)
  "y. But now let's switch gears."
  ' the blueprint for that trust.'
  ' it went public in April 2019.'


✍️ **Answer — Q1 (sentence boundaries):**

Out of the box: **a little better, not "solved."** With the default separators (no sentence
punctuation in the list at all), `RecursiveCharacterTextSplitter` gets **10.8%** clean endings on
the podcast and **22.4%** on the PDF at `chunk_size=500` — both higher than plain
`CharacterTextSplitter(separator="")` from Step 2 (5.4% / 3.2%), because it stays word-safe
(and line-safe when lines exist) rather than cutting raw characters. But it's still nowhere near
reliable, because **no sentence-ending separator is in the default list** — it can only ever land
on a `.`/`!`/`?` by coincidence.

Adding one yourself has a real gotcha. Attempt A adds `". "`, `"! "`, `"? "` to the separator list
but keeps the default `keep_separator=False` — result: **1/37 (2.7%) clean**, *worse* than doing
nothing. Why: `RecursiveCharacterTextSplitter` splits with `re.split()` on the literal separator
and, without `keep_separator`, **throws the separator away**. Every sentence loses its trailing
`.`/`!`/`?` when it's split off, and it's only restored on fragments that get *merged* with another
fragment afterward — the fragment that ends up last in a chunk never gets it back. Attempt B adds
`keep_separator="end"` (which reattaches the matched separator to the end of each piece instead of
discarding it) and jumps to **37/37 (100%) clean** — a complete fix, using the exact same separator
list.

**Takeaway:** recursive chunking preserves sentences better than fixed-size chunking only if a
sentence-ending separator is explicitly added to the list *and* `keep_separator` is set correctly.
Out of the box, it only guarantees word (and sometimes line) safety — never sentence safety — and
a careless attempt to add sentence-awareness can silently make things worse, not better.


**2. How does it handle the podcast's conversational structure?**

In [14]:
print("podcast '\\n\\n' count:", podcast_text.count("\n\n"))
print("podcast '\\n' count:  ", podcast_text.count("\n"))

# With neither \n\n nor \n present, which separator does the default list actually land on?
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
recursive_chunks = splitter.split_text(podcast_text)
fixed_chunks = CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50).split_text(podcast_text)

sizes_r = [len(c) for c in recursive_chunks]
sizes_f = [len(c) for c in fixed_chunks]
print(f"\nrecursive (default): n={len(recursive_chunks)}, avg={statistics.mean(sizes_r):.1f}, max={max(sizes_r)}")
print(f"fixed (separator=''): n={len(fixed_chunks)}, avg={statistics.mean(sizes_f):.1f}, max={max(sizes_f)}")


podcast '\n\n' count: 0
podcast '\n' count:   0

recursive (default): n=37, avg=489.5, max=499
fixed (separator=''): n=37, avg=493.1, max=500


✍️ **Answer — Q2 (podcast's conversational structure):**

Badly — because there's no structure in the transcript for it to find. The podcast has **0**
occurrences of `\n\n` and **0** occurrences of plain `\n` (it's one unbroken block of
speech-to-text, with no paragraph or line breaks at all). `RecursiveCharacterTextSplitter`'s
algorithm checks each separator in its priority list in order and commits to the **first one
that actually appears in the text** — since neither `\n\n` nor `\n` appears, it skips straight
past both and lands on `" "` (word boundaries) for the entire transcript.

That makes it, for this text, almost indistinguishable from plain `CharacterTextSplitter`: 37
chunks either way, nearly identical average size (489.5 vs. 493.1 chars) and max (499 vs. 500).
The only real difference is that `separator=" "` guarantees every cut falls between two words
(never inside one), which `separator=""` doesn't — but since conversational speech rarely has
long unbroken "words," that guarantee barely changes the outcome in practice.

**Takeaway:** "recursive" chunking only buys you something when the text actually *has* the
structure the separator list is built to exploit (paragraphs, lines). A transcript like this one
— one continuous block with no formatting — defeats the whole point of the priority list; it
degrades to the simplest level that still matches anything (word splitting), which is barely
better than character splitting.


**3. Does it respect PDF section headers?**

In [15]:
# Find a few real section headers in the PDF (short, all-caps lines)
pdf_lines = pdf_text.split("\n")
headings = [l.strip() for l in pdf_lines if 5 < len(l.strip()) < 40 and l.strip() == l.strip().upper() and l.strip().isupper()]
sample_headings = ["EXECUTIVE SUMMARY", "TABLE OF CONTENTS", "GLOSSARY", "D. CONCLUSION"]
print("example heading lines found in the PDF:", headings[:6])
print()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
rec_chunks = splitter.split_text(pdf_text)
fixed_chunks_pdf = CharacterTextSplitter(separator="", chunk_size=1000, chunk_overlap=50).split_text(pdf_text)

print(f"{'heading':22s} {'recursive: chunk, offset/len':30s} fixed (sep=''): chunk, offset/len")
for h in sample_headings:
    r_info = next((f"#{i}, {c.find(h)}/{len(c)}" for i, c in enumerate(rec_chunks) if h in c), "not found")
    f_info = next((f"#{i}, {c.find(h)}/{len(c)}" for i, c in enumerate(fixed_chunks_pdf) if h in c), "not found")
    print(f"{h:22s} {r_info:30s} {f_info}")


example heading lines found in the PDF: ['INDEPENDENT', 'HIGH-LEVEL EXPERT GROUP ON', 'ARTIFICIAL INTELLIGENCE', 'SET UP BY THE EUROPEAN COMMISSION', 'ETHICS GUIDELINES', 'FOR TRUSTWORTHY AI']



heading                recursive: chunk, offset/len   fixed (sep=''): chunk, offset/len
EXECUTIVE SUMMARY      #2, 703/936                    #2, 694/996
TABLE OF CONTENTS      #2, 680/936                    #2, 671/996
GLOSSARY               #3, 223/955                    #3, 202/1000
D. CONCLUSION          #3, 205/955                    #3, 184/1000


✍️ **Answer — Q3 (PDF section headers):**

No — not meaningfully. For 4 real section headers ("EXECUTIVE SUMMARY", "TABLE OF CONTENTS",
"GLOSSARY", "D. CONCLUSION"), `RecursiveCharacterTextSplitter(chunk_size=1000)` buries every one
of them **mid-chunk** (offsets 205–703 inside 936–955-character chunks) — never at the start of a
chunk. Plain `CharacterTextSplitter(separator="")` lands them at almost the *same* offsets
(184–694 inside 996–1000-char chunks). The two strategies are barely distinguishable here.

The reason: the PDF has 1,961 occurrences of `\n` but only 3 of `\n\n`, so the recursive algorithm
picks `\n\n` first (it's present), producing just 4 giant initial pieces; each one then recurses
into `\n` (also present, abundant), splitting into **individual lines** — including heading lines,
but *not treating them any differently from body-text lines*. The merge step then just packs
whole lines greedily up to `chunk_size`, stopping only when the next line would overflow — not
when the next line "looks like a heading." A heading only ever starts a fresh chunk by
coincidence, if the line before it happened to fill the chunk exactly.

**Takeaway:** `RecursiveCharacterTextSplitter` guarantees it won't cut a *line* in half, but it
has no concept of "this line is structurally special" (a heading, a list marker) versus "this
line is a sentence from a paragraph" — both are just lines to be packed. Actually respecting
section headers would need either a different library (one that parses document structure, e.g.
headers from Markdown/HTML) or a custom separator that specifically targets heading patterns
(e.g. a regex for short, all-caps, or numbered lines) placed *above* `"\n"` in the priority list.


# Step 4 — Implement Token-Based Chunking

## 1. Token-based chunks for both documents with `TokenTextSplitter`.

In [16]:
import tiktoken

splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=300, chunk_overlap=50)
pdf_token_chunks = splitter.split_text(pdf_text)
podcast_token_chunks = splitter.split_text(podcast_text)
print(len(pdf_token_chunks), "PDF chunks,", len(podcast_token_chunks), "podcast chunks")

# Look at a boundary directly -- same check we did for CharacterTextSplitter in Step 2
enc = tiktoken.get_encoding("cl100k_base")
for i in (0, 1, 2):
    print(f"chunk {i} end:   ...{podcast_token_chunks[i][-50:]!r}")
    print(f"chunk {i+1} start: {podcast_token_chunks[i+1][:50]!r}...\n")


127 PDF chunks, 14 podcast chunks
chunk 0 end:   ...'is document, it went public in April 2019. In tech'
chunk 1 start: ' an independent group set up by the European Commi'...

chunk 1 end:   ...' of it like a three-legged stool. The first leg is'
chunk 2 start: '.I. as having three components. Right. And they ha'...

chunk 2 end:   ...' So lawful, ethical, and robust. The document then'
chunk 3 start: ' A.I., but if it crashes every time it sees a pixe'...



## Analysis questions

Same rule as Step 2: answer with **numbers from your own runs**, not "yes"/"no".

**1. Does token-based chunking also break words or sentences in the middle?**

In [17]:
# Count how many chunks end "cleanly" (on sentence punctuation), varying chunk_size / overlap
print(f"{'size':>6} {'overlap':>8} {'n_chunks':>9} {'clean_end':>10} {'pct_clean':>10}")
for chunk_size in (150, 300, 500):
    for overlap in (0, 50, 100):
        splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=chunk_size, chunk_overlap=overlap)
        chunks = splitter.split_text(podcast_text)
        clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
        pct_clean = 100 * clean_end / len(chunks)
        print(f"{chunk_size:6d} {overlap:8d} {len(chunks):9d} {clean_end:10d} {pct_clean:9.1f}%")


  size  overlap  n_chunks  clean_end  pct_clean
   150        0        24          1       4.2%
   150       50        35          2       5.7%
   150      100        69          4       5.8%
   300        0        12          1       8.3%
   300       50        14          2      14.3%
   300      100        18          2      11.1%
   500        0         8          1      12.5%
   500       50         8          2      25.0%
   500      100         9          1      11.1%


✍️ **Answer — Q1 (word/sentence breaks):**

Yes. The boundary print above shows it directly: chunk 0 ends "...public in April 2019. In tech" and
chunk 1 starts " an independent group set up..." — a word ("tech[nology]") and a sentence cut mid-way,
exactly like `CharacterTextSplitter`.

Counting it: with `chunk_size=300, chunk_overlap=50`, only **2 of 14** podcast chunks (14.3%) end
cleanly on `.`, `?` or `!`. This is actually a bit *higher* than Step 2's character-based result at a
comparable size (5.4% at 500 chars), but it stays in the same ballpark and the same range as size/overlap
change — from **4.2%** (size 150, overlap 0) up to **25.0%** (size 500, overlap 50). The reason it isn't
zero is coincidence: `TokenTextSplitter` cuts by **token position**, with no concept of a sentence or
even a word — a token boundary landing right after a `.` token is luck, not design, same as a character
count landing on a `.` was luck in Step 2.

**Why it's not identical to character-based breaking:** tokens (BPE subword units) don't always split
on character runs the same way characters do — a word like "chunking" can be one token or two
("chunk" + "ing"), so a cut can fall *inside* a word at a point a character-count cut would never
naturally land on, or it can happen to land exactly on a word boundary because that word happened to
be a single token. Either way, the splitter has no sentence-awareness, so the structural problem from
Step 2 — chunk boundaries that ignore meaning — is identical in kind, even though the exact cut points
differ.


**2. How does token-based chunking compare to character-based chunking (Step 2)?**

> 💡 A token isn't a fixed number of characters — tokenizers use variable-length subword units.
> So "300 tokens" and "500 characters" aren't directly comparable sizes. To compare fairly, convert
> using the text's own **characters-per-token ratio**, then check: for the *same* target size, which
> splitter actually keeps the token count under control (the thing that matters for an LLM's context
> window), and which keeps the character count more consistent?

In [18]:
for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
    ratio = len(text) / len(enc.encode(text))  # chars per token, specific to this text

    tok_splitter = TokenTextSplitter(encoding_name="cl100k_base", chunk_size=300, chunk_overlap=50)
    tok_chunks = tok_splitter.split_text(text)

    # same target size, converted to characters, so the comparison is fair
    char_size = round(300 * ratio)
    char_overlap = round(50 * ratio)
    char_splitter = CharacterTextSplitter(separator="", chunk_size=char_size, chunk_overlap=char_overlap)
    char_chunks = char_splitter.split_text(text)

    tok_tokens = [len(enc.encode(c)) for c in tok_chunks]
    char_tokens = [len(enc.encode(c)) for c in char_chunks]
    tok_chars = [len(c) for c in tok_chunks]
    char_chars = [len(c) for c in char_chunks]

    print(f"{label}  (ratio={ratio:.2f} chars/token, char-equivalent target={char_size} chars)")
    print(f"  token-based:  n={len(tok_chunks):4d}  tokens[min,max]=({min(tok_tokens)},{max(tok_tokens)})   "
          f"chars[min,max]=({min(tok_chars)},{max(tok_chars)})")
    print(f"  char-based:   n={len(char_chunks):4d}  tokens[min,max]=({min(char_tokens)},{max(char_tokens)})   "
          f"chars[min,max]=({min(char_chars)},{max(char_chars)})")
    print()


podcast  (ratio=4.69 chars/token, char-equivalent target=1408 chars)
  token-based:  n=  14  tokens[min,max]=(257,300)   chars[min,max]=(1089,1503)
  char-based:   n=  14  tokens[min,max]=(281,322)   chars[min,max]=(1212,1408)



pdf  (ratio=4.93 chars/token, char-equivalent target=1480 chars)
  token-based:  n= 127  tokens[min,max]=(210,300)   chars[min,max]=(748,1724)
  char-based:   n= 127  tokens[min,max]=(256,425)   chars[min,max]=(1050,1480)



✍️ **Answer — Q2 (token-based vs. character-based):**

Converting fairly (300 tokens ≈ 1,408 characters for the podcast, ≈1,480 for the PDF, using each
text's own chars/token ratio) and comparing both splitters at that equivalent target:

- **Podcast:** token-based → 14 chunks, tokens range **257–300**, chars range 1,089–1,503.
  Char-based (same target) → 14 chunks, tokens range **281–322**, chars range 1,212–1,408.
- **PDF:** token-based → 127 chunks, tokens range **210–300**, chars range 748–1,724.
  Char-based (same target) → 127 chunks, tokens range **256–425**, chars range 1,050–1,480.

Both produce roughly the same **number** of chunks for the same target size — not surprising, since
both are mechanically "fixed window, slide forward." The real difference is **which unit stays under
control**: `TokenTextSplitter` guarantees every chunk is **at most 300 tokens** (that's the whole
point — it counts the actual unit an LLM's context window is measured in), while its character count
swings more (1,089–1,724 across both texts). `CharacterTextSplitter` is the mirror image: its
character count is tightly capped by `chunk_size`, but its **token count is not controlled at all** —
on the PDF it overshoots to **425 tokens**, 42% over the 300-token budget we were aiming for, because
some stretches of text use shorter, more token-dense wording than others.

**Takeaway:** if the downstream goal is "fit inside an LLM's context window" or "use roughly even API
cost per chunk," `TokenTextSplitter` is the right tool — it directly controls the unit that matters.
`CharacterTextSplitter` only controls a proxy for that unit, and the proxy can be off by more than 40%.


**3. Which content type handles token-based chunking better?**

In [19]:
for label, chunks in (("podcast", podcast_token_chunks), ("pdf", pdf_token_chunks)):
    sizes_chars = [len(c) for c in chunks]
    sizes_tokens = [len(enc.encode(c)) for c in chunks]
    clean_end = sum(1 for c in chunks if c.rstrip().endswith((".", "?", "!")))
    print(f"{label}: n={len(chunks)}, avg_chars={statistics.mean(sizes_chars):.1f}, "
          f"min_chars={min(sizes_chars)}, max_chars={max(sizes_chars)}, "
          f"avg_tokens={statistics.mean(sizes_tokens):.1f}, max_tokens={max(sizes_tokens)}, "
          f"clean_end={clean_end} ({100*clean_end/len(chunks):.1f}%)")


podcast: n=14, avg_chars=1386.9, min_chars=1089, max_chars=1503, avg_tokens=296.9, max_tokens=300, clean_end=2 (14.3%)
pdf: n=127, avg_chars=1477.9, min_chars=748, max_chars=1724, avg_tokens=299.3, max_tokens=300, clean_end=8 (6.3%)


✍️ **Answer — Q3 (podcast vs. PDF):**

At `chunk_size=300, chunk_overlap=50`: the podcast yields **14 chunks** (avg 1,386.9 chars /
296.9 tokens, max 300 tokens) with **14.3%** clean sentence endings. The PDF yields **127
chunks** (avg 1,477.9 chars / 299.3 tokens, max 300 tokens) with only **6.3%** clean endings —
the same ordering as Step 2's `CharacterTextSplitter` result (5.4% vs. 3.2%), for the same
underlying reason: the PDF has more headings, bullet lists and footnotes relative to its length,
so there's less sentence-ending punctuation per chunk for a token (or character) cut to land on
by chance.

What token-based chunking *does* fix, that character-based chunking never could: **both texts
stay almost exactly at the 300-token cap** (296.9 and 299.3 tokens on average, never over 300) —
compare that to Step 2, where `separator="\n\n"` let the PDF balloon to a 74,680-character chunk
with no limit at all. So for *size control*, token-based chunking handles both content types
equally well and reliably, regardless of their structure. For *boundary quality* (not cutting
mid-sentence), neither content type is handled "well" by `TokenTextSplitter` — it has no more
sentence-awareness than `CharacterTextSplitter`'s `separator=""` did, and the PDF is still the
harder text for that specific problem, same as in Step 2.


# Step 7 — Analyze Chunk Quality

**Do:** Evaluate chunk quality by checking boundary preservation, across *every* strategy used
so far, on both documents.

In [20]:
def pct_clean_boundaries(chunks):
    clean = sum(1 for c in chunks if c.strip()[-1] in ".?!")
    return clean / len(chunks) * 100

# Recompute every strategy fresh, at comparable target sizes (500 chars / ~300 tokens),
# so this table doesn't depend on which cell ran last earlier in the notebook.
strategies = {
    'Fixed-500 (sep="")': lambda t: CharacterTextSplitter(separator="", chunk_size=500, chunk_overlap=50).split_text(t),
    'Fixed-500 (default "\\n\\n")': lambda t: CharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_text(t),
    "Recursive-500 (default)": lambda t: RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).split_text(t),
    "Recursive-500 (sentence-aware)": lambda t: RecursiveCharacterTextSplitter(
        chunk_size=500, chunk_overlap=50,
        separators=["\n\n", "\n", ". ", "! ", "? ", " ", ""], keep_separator="end",
    ).split_text(t),
    "Token-300": lambda t: TokenTextSplitter(encoding_name="cl100k_base", chunk_size=300, chunk_overlap=50).split_text(t),
    "Fixed-500, regex [.!?\\n]\\s+": lambda t: CharacterTextSplitter(
        separator=r"[.!?\n]\s+", is_separator_regex=True, keep_separator="end",
        chunk_size=500, chunk_overlap=50,
    ).split_text(t),
}

rows = []
for name, fn in strategies.items():
    for label, text in (("podcast", podcast_text), ("pdf", pdf_text)):
        chunks = fn(text)
        sizes = [len(c) for c in chunks]
        rows.append({
            "strategy": name, "text": label, "n_chunks": len(chunks),
            "avg_chars": round(statistics.mean(sizes), 1),
            "max_chars": max(sizes),
            "pct_clean": round(pct_clean_boundaries(chunks), 1),
        })

quality_df = pd.DataFrame(rows)
quality_df


Created a chunk of size 28061, which is longer than the specified 500


Created a chunk of size 29967, which is longer than the specified 500


Created a chunk of size 23694, which is longer than the specified 500


Created a chunk of size 602, which is longer than the specified 500


Created a chunk of size 546, which is longer than the specified 500


...

Created a chunk of size 856, which is longer than the specified 500


Created a chunk of size 735, which is longer than the specified 500


Created a chunk of size 512, which is longer than the specified 500


Created a chunk of size 683, which is longer than the specified 500


,strategy,text,n_chunks,avg_chars,max_chars,pct_clean
0,"Fixed-500 (sep="""")",podcast,37,493.1,500,5.4
1,"Fixed-500 (sep="""")",pdf,348,498.0,500,3.2
2,"Fixed-500 (default ""\n\n"")",podcast,1,16461.0,16461,100.0
3,"Fixed-500 (default ""\n\n"")",pdf,4,39093.0,74680,50.0
4,Recursive-500 (default),podcast,37,489.5,499,10.8
5,Recursive-500 (default),pdf,353,443.2,498,22.4
6,Recursive-500 (sentence-aware),podcast,37,462.4,498,100.0
7,Recursive-500 (sentence-aware),pdf,353,443.2,498,22.4
8,Token-300,podcast,14,1386.9,1503,14.3
9,Token-300,pdf,127,1477.9,1724,6.3


✍️ **Which strategy preserves context best, and why?**

The regex-based `CharacterTextSplitter` (`separator=r"[.!?\n]\s+"`, `is_separator_regex=True`,
`keep_separator="end"`) wins outright on both texts — **100% clean endings on the podcast, 92.7%
on the PDF** — beating every other strategy tested, including the "recommended"
`RecursiveCharacterTextSplitter`, which only reaches 10.8%/22.4% with its default separators and
*doesn't improve at all on the PDF* (still 22.4%) even after we gave it a sentence-aware separator
list in Step 3 Q1.

The reason is structural, not incidental: `RecursiveCharacterTextSplitter` commits to a strict
**hierarchy** — it picks the first separator in its list that appears anywhere in the text, uses
it for the whole piece, and only drops to the next separator for pieces still too big. The PDF is
full of single `\n` line breaks, so the recursion locks onto line-splitting before it ever gets a
chance to also consider `.`/`!`/`?` as valid cut points — even though most lines don't end on a
sentence. The regex `[.!?\n]\s+` has no hierarchy: it treats punctuation and line breaks as
**equally valid split points in one pass**, so it can stop at whichever boundary comes first,
sentence or line. That one design difference is why a single flat regex check beats a
"smarter," recommended multi-level splitter on real structured text.


# Step 8 — Make Recommendations

## Chunking Strategy Recommendations

### For PDF Documents:
**Recommended Strategy:** `CharacterTextSplitter` with a combined regex separator
(`r"[.!?\n]\s+"`, `is_separator_regex=True`, `keep_separator="end"`), `chunk_size=500`,
`chunk_overlap=50`.

**Reasoning:**
- It reaches **92.7%** clean sentence/line endings on this PDF — 4x `RecursiveCharacterTextSplitter`'s
  default (22.4%), and its sentence-aware variant doesn't do any better (also 22.4%), because the
  PDF's abundant single `\n` breaks make the recursive hierarchy commit to line-splitting before
  it ever considers sentence punctuation (see Step 3 Q1/Q3, Step 7).
- Key advantage: it treats headings/line breaks *and* sentence punctuation as equally valid cut
  points in one pass, instead of prioritizing one over the other — which matters for a document
  that mixes prose paragraphs with headings, lists and footnotes.
- Optimal size/overlap found here: `chunk_size=500`, `chunk_overlap=50`. `chunk_size` is a soft
  cap with this approach (9 of 385 chunks ran over 500 chars, up to 959) — acceptable, since it's
  still far tighter than the 74,680-character runaway chunk the plain `"\n\n"` separator produced
  in Step 2.

### For Podcast Transcripts:
**Recommended Strategy:** depends on the downstream consumer — two valid answers, not one:

- **If the consumer is an LLM with a hard context/cost budget:** `TokenTextSplitter`
  (`cl100k_base`, `chunk_size=300`, `chunk_overlap=50`). It's the *only* strategy tested that
  directly caps the unit that actually matters (tokens): every chunk stayed at or under 300
  tokens, while `CharacterTextSplitter` at an equivalent character target overshot to 425 tokens
  on the PDF (Step 4 Q2) — a 42% budget overrun a token-blind splitter can't prevent.
- **If the consumer cares about retrieval/semantic quality more than exact size:** the same
  regex-based `CharacterTextSplitter` as the PDF (`r"[.!?\n]\s+"`, `keep_separator="end"`) — it
  hits **100%** clean sentence endings on the podcast, because the transcript has no `\n` at all,
  so the recursive-style hierarchy problem never applies here.
- **Optimal chunk size and overlap:** 300 tokens / 50 overlap for the token-budget case; 500
  characters / 50 overlap for the sentence-quality case. Either way, some overlap matters —
  without it, a reader loses the sentence that straddles two chunks entirely.

### Trade-offs Summary:
| Strategy | Pros | Cons | Best For |
|----------|------|------|----------|
| Fixed-500 (`sep=""`) | Simplest; always obeys `chunk_size` exactly | No structural awareness at all; worst clean-ending rate measured (5.4% / 3.2%) | Quick prototyping on any text |
| Recursive-500 (default separators) | Never cuts mid-word; respects lines when present | No sentence awareness; identical to naive character-splitting on the PDF (22.4%) because the hierarchy locks onto `\n` | Lightly structured text where lines are already natural units |
| Recursive-500 (custom sentence separators) | Large win when the text has no finer separator already present (podcast: 2.7% → 100%) | Zero effect when a higher-priority separator (`\n`) is already everywhere — the PDF result doesn't move | Plain prose/transcripts with no line breaks |
| Token-300 | Only strategy that directly caps the context-window/cost unit | No sentence awareness (14.3% / 6.3%); character length varies a lot chunk to chunk | Feeding a hard LLM context-window or cost budget |
| Fixed-500, regex `[.!?\n]\s+` | Best clean-ending result on **both** texts (100% / 92.7%) by treating every boundary type as one flat choice instead of a hierarchy | `chunk_size` becomes a soft cap (up to 959 chars on the PDF); misses abbreviations like "A.I." | Retrieval/RAG quality, where boundary cleanliness matters more than an exact size cap |
